<a href="https://colab.research.google.com/github/akshaykrishnas2200-cpu/Machine-learning/blob/main/exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

In [2]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']
print("Loading 20 Newsgroups dataset...")
train_data = fetch_20newsgroups(subset='train', categories=categories,
remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', categories=categories,
remove=('headers', 'footers', 'quotes'))

Loading 20 Newsgroups dataset...


In [3]:
vectorizer = CountVectorizer(stop_words='english', max_features=5000)
X_train = vectorizer.fit_transform(train_data.data).toarray()
y_train = train_data.target
X_test = vectorizer.transform(test_data.data).toarray()
y_test = test_data.target

In [4]:
n_classes = len(categories)
n_features = X_train.shape[1]
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
# Calculate class priors P(c)
class_counts = np.bincount(y_train)
p_labels = class_counts / len(y_train)

Train shape: (2257, 5000), Test shape: (1502, 5000)


In [6]:
N_c = np.zeros((n_classes, n_features))
for c in range(n_classes):
    N_c[c, :] = X_train[y_train == c].sum(axis=0)
print("Word counts aggregated successfully.")
def predict_naive_bayes(X, theta, p_labels):
    """
    Predicts classes using log-probabilities to avoid underflow.
    Log-likelihood: X . log(Theta^T) + log(P(c))
    """
    with np.errstate(divide='ignore'):
        log_theta = np.log(theta)

Word counts aggregated successfully.


In [8]:
theta_mle = N_c / N_c.sum(axis=1, keepdims=True)

with np.errstate(divide='ignore'):
    log_theta = np.log(theta_mle)

log_theta[np.isinf(log_theta)] = -1e9

# Using X_train for demonstration as X is an argument to predict_naive_bayes
X = X_train

log_posterior = X @ log_theta.T + np.log(p_labels)

# Removed 'return' statement as this code is outside a function
predictions = np.argmax(log_posterior, axis=1)
print(f"Sample predictions (first 5): {predictions[:5]}")

Sample predictions (first 5): [1 1 3 3 3]


In [10]:
# Reimplementing the prediction logic that predict_naive_bayes should have performed
with np.errstate(divide='ignore'):
    log_theta_mle_for_prediction = np.log(theta_mle)
log_theta_mle_for_prediction[np.isinf(log_theta_mle_for_prediction)] = -1e9
log_posterior_mle = X_test @ log_theta_mle_for_prediction.T + np.log(p_labels)
y_pred_mle = np.argmax(log_posterior_mle, axis=1)

mle_accuracy = accuracy_score(y_test, y_pred_mle)
print(f"[MLE] Test Accuracy: {mle_accuracy:.4f}")
priors_to_test = {
"Lidstone Smoothing (Alpha = 1.01)": np.ones(n_features) * 1.01,
"Laplace Smoothing (Alpha = 2.0)": np.ones(n_features) * 2.0,
"Strong Dirichlet Prior (Alpha = 10.0)": np.ones(n_features) * 10.0,
"Non-Uniform Prior (Empirical Basis)": 1.0 + (X_train.sum(axis=0) / X_train.sum() *
100)
}

[MLE] Test Accuracy: 0.7710


In [12]:
print("Evaluating MAP Predictions:")
print("-" * 55)
for name, alpha in priors_to_test.items():
    numerator = N_c + (alpha - 1)
    denominator = numerator.sum(axis=1, keepdims=True)
    theta_map = numerator / denominator

    # Reimplementing the prediction logic that predict_naive_bayes should have performed
    with np.errstate(divide='ignore'):
        log_theta_map_for_prediction = np.log(theta_map)
    log_theta_map_for_prediction[np.isinf(log_theta_map_for_prediction)] = -1e9
    log_posterior_map = X_test @ log_theta_map_for_prediction.T + np.log(p_labels)
    y_pred_map = np.argmax(log_posterior_map, axis=1)

    map_accuracy = accuracy_score(y_test, y_pred_map)
    print(f"{name:38} | Test Accuracy: {map_accuracy:.4f}")

Evaluating MAP Predictions:
-------------------------------------------------------
Lidstone Smoothing (Alpha = 1.01)      | Test Accuracy: 0.8103
Laplace Smoothing (Alpha = 2.0)        | Test Accuracy: 0.8182
Strong Dirichlet Prior (Alpha = 10.0)  | Test Accuracy: 0.7863
Non-Uniform Prior (Empirical Basis)    | Test Accuracy: 0.8063
